# 05 — Modelo de Redes Neurais com MLlib

Este notebook treina modelos de **Redes Neurais** usando **PySpark MLlib** para classificar a faixa térmica futura no estado de São Paulo.

Diferente dos notebooks de Regressão Linear e Random Forest, que preveem temperatura em graus Celsius, aqui o problema será tratado como **classificação**.

Serão treinados dois modelos:

1. **Classificação da faixa térmica de amanhã**.
2. **Classificação da faixa térmica média dos próximos 7 dias**.

Este notebook mantém o mesmo padrão dos notebooks anteriores:

- uso de **MLlib** para o modelo;
- uso de **Spark SQL** para consultas, criação de alvos, joins, agregações e análises;
- uso de **Plotly** para gráficos;
- sem uso de `toPandas()`.

O modelo utilizado será o **Multilayer Perceptron Classifier**, uma rede neural disponível no MLlib para tarefas de classificação.

## 1. Por que transformar temperatura em categorias?

O PySpark MLlib possui uma rede neural nativa chamada `MultilayerPerceptronClassifier`.

Esse modelo é voltado para **classificação**, não para regressão contínua. Como a variável temperatura é originalmente numérica, será criada uma versão categórica do alvo.

Assim, em vez de responder:

> qual será a temperatura exata em °C?

a rede neural responderá:

> qual será a faixa térmica esperada?

As faixas usadas neste notebook serão definidas com base nos quartis da base de treino.

Essa escolha evita criar classes muito desbalanceadas e reduz o risco de o modelo aprender apenas a classe majoritária.

Os cortes são calculados separadamente para cada objetivo:

- `temperatura_amanha`;
- `temperatura_media_proximos_7_dias`.

As classes serão:

| Classe numérica | Categoria | Interpretação |
|---:|---|---|
| 0 | mais_fria | valores abaixo do 1º quartil |
| 1 | amena | valores entre o 1º e o 2º quartil |
| 2 | quente | valores entre o 2º e o 3º quartil |
| 3 | mais_quente | valores acima do 3º quartil |

Os quartis são calculados apenas na base de treino e depois aplicados também na base de teste, evitando vazamento de dados.

## 2. Imports e inicialização da SparkSession

In [95]:
from pyspark.sql import SparkSession

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

import plotly.express as px
import plotly.graph_objects as go

In [96]:
spark = (
    SparkSession.builder
    .appName("05_modelo_redes_neurais_mllib")
    .getOrCreate()
)

spark

## 3. Caminhos das bases Parquet

As bases finais já foram criadas no notebook de pré-processamento.

Também serão carregadas as bases completas, porque elas ainda possuem colunas interpretáveis como `tipo_area`, `macro_regiao_sp` e `faixa_altitude`.

In [97]:
amanha_train_path = "/home/jovyan/work/data/processed/weather_sp_amanha_train"
amanha_test_path  = "/home/jovyan/work/data/processed/weather_sp_amanha_test"

semana_train_path = "/home/jovyan/work/data/processed/weather_sp_semana_train"
semana_test_path  = "/home/jovyan/work/data/processed/weather_sp_semana_test"

dataset_amanha_path = "/home/jovyan/work/data/processed/weather_sp_dataset_amanha"
dataset_semana_path = "/home/jovyan/work/data/processed/weather_sp_dataset_semana"

modelos_path = "/home/jovyan/work/models"
resultados_path = "/home/jovyan/work/data/processed"

## 4. Carregamento das bases

As bases são lidas diretamente do formato Parquet e registradas como views temporárias para uso com `spark.sql`.

In [98]:
amanha_train = spark.read.parquet(amanha_train_path)
amanha_test = spark.read.parquet(amanha_test_path)

semana_train = spark.read.parquet(semana_train_path)
semana_test = spark.read.parquet(semana_test_path)

dataset_amanha_completo = spark.read.parquet(dataset_amanha_path)
dataset_semana_completo = spark.read.parquet(dataset_semana_path)

amanha_train.createOrReplaceTempView("amanha_train")
amanha_test.createOrReplaceTempView("amanha_test")
semana_train.createOrReplaceTempView("semana_train")
semana_test.createOrReplaceTempView("semana_test")
dataset_amanha_completo.createOrReplaceTempView("dataset_amanha_completo")
dataset_semana_completo.createOrReplaceTempView("dataset_semana_completo")

In [99]:
spark.sql("""
    SELECT 'amanha_train' AS base, COUNT(*) AS total_linhas FROM amanha_train
    UNION ALL
    SELECT 'amanha_test' AS base, COUNT(*) AS total_linhas FROM amanha_test
    UNION ALL
    SELECT 'semana_train' AS base, COUNT(*) AS total_linhas FROM semana_train
    UNION ALL
    SELECT 'semana_test' AS base, COUNT(*) AS total_linhas FROM semana_test
""").show(truncate=False)

+------------+------------+
|base        |total_linhas|
+------------+------------+
|amanha_train|120232      |
|amanha_test |45940       |
|semana_train|120232      |
|semana_test |45940       |
+------------+------------+



## 5. Cache estratégico dos datasets de modelagem

No pré-processamento, o cache foi evitado para reduzir consumo de memória.

Aqui, no notebook de modelagem, o cache faz sentido porque os mesmos datasets serão usados várias vezes.

In [100]:
amanha_train = amanha_train.cache()
amanha_test = amanha_test.cache()
semana_train = semana_train.cache()
semana_test = semana_test.cache()

amanha_train.count()
amanha_test.count()
semana_train.count()
semana_test.count()

amanha_train.createOrReplaceTempView("amanha_train")
amanha_test.createOrReplaceTempView("amanha_test")
semana_train.createOrReplaceTempView("semana_train")
semana_test.createOrReplaceTempView("semana_test")

## 6. Conferência dos schemas

In [101]:
amanha_train.printSchema()
semana_train.printSchema()

root
 |-- station: string (nullable = true)
 |-- station_code: string (nullable = true)
 |-- data_formatada: date (nullable = true)
 |-- ano_imputado: integer (nullable = true)
 |-- mes_sin_imputado: double (nullable = true)
 |-- mes_cos_imputado: double (nullable = true)
 |-- latitude_imputado: double (nullable = true)
 |-- longitude_imputado: double (nullable = true)
 |-- altitude_imputado: double (nullable = true)
 |-- temp_media_dia_imputado: double (nullable = true)
 |-- temp_min_dia_imputado: double (nullable = true)
 |-- temp_max_dia_imputado: double (nullable = true)
 |-- temp_orvalho_media_dia_imputado: double (nullable = true)
 |-- umidade_media_dia_imputado: double (nullable = true)
 |-- umidade_min_dia_imputado: double (nullable = true)
 |-- umidade_max_dia_imputado: double (nullable = true)
 |-- pressao_media_dia_imputado: double (nullable = true)
 |-- precipitacao_total_dia_imputado: double (nullable = true)
 |-- radiacao_media_dia_imputado: double (nullable = true)
 |-- 

## 7. Criação dos alvos categóricos com Spark SQL

Nesta etapa, os alvos numéricos são transformados em classes térmicas.

In [102]:
# Cálculo dos quartis apenas na base de treino
q1_amanha, q2_amanha, q3_amanha = amanha_train.approxQuantile(
    "temperatura_amanha",
    [0.25, 0.50, 0.75],
    0.01
)

q1_semana, q2_semana, q3_semana = semana_train.approxQuantile(
    "temperatura_media_proximos_7_dias",
    [0.25, 0.50, 0.75],
    0.01
)

print("Cortes para temperatura de amanhã:")
print(f"Q1: {q1_amanha:.2f}")
print(f"Q2: {q2_amanha:.2f}")
print(f"Q3: {q3_amanha:.2f}")

print("\nCortes para temperatura média dos próximos 7 dias:")
print(f"Q1: {q1_semana:.2f}")
print(f"Q2: {q2_semana:.2f}")
print(f"Q3: {q3_semana:.2f}")

amanha_train_cls = spark.sql(f"""
    SELECT *,
        CASE
            WHEN temperatura_amanha < {q1_amanha} THEN CAST(0 AS DOUBLE)
            WHEN temperatura_amanha < {q2_amanha} THEN CAST(1 AS DOUBLE)
            WHEN temperatura_amanha < {q3_amanha} THEN CAST(2 AS DOUBLE)
            ELSE CAST(3 AS DOUBLE)
        END AS classe_temperatura_amanha
    FROM amanha_train
""")

amanha_test_cls = spark.sql(f"""
    SELECT *,
        CASE
            WHEN temperatura_amanha < {q1_amanha} THEN CAST(0 AS DOUBLE)
            WHEN temperatura_amanha < {q2_amanha} THEN CAST(1 AS DOUBLE)
            WHEN temperatura_amanha < {q3_amanha} THEN CAST(2 AS DOUBLE)
            ELSE CAST(3 AS DOUBLE)
        END AS classe_temperatura_amanha
    FROM amanha_test
""")

semana_train_cls = spark.sql(f"""
    SELECT *,
        CASE
            WHEN temperatura_media_proximos_7_dias < {q1_semana} THEN CAST(0 AS DOUBLE)
            WHEN temperatura_media_proximos_7_dias < {q2_semana} THEN CAST(1 AS DOUBLE)
            WHEN temperatura_media_proximos_7_dias < {q3_semana} THEN CAST(2 AS DOUBLE)
            ELSE CAST(3 AS DOUBLE)
        END AS classe_temperatura_semana
    FROM semana_train
""")

semana_test_cls = spark.sql(f"""
    SELECT *,
        CASE
            WHEN temperatura_media_proximos_7_dias < {q1_semana} THEN CAST(0 AS DOUBLE)
            WHEN temperatura_media_proximos_7_dias < {q2_semana} THEN CAST(1 AS DOUBLE)
            WHEN temperatura_media_proximos_7_dias < {q3_semana} THEN CAST(2 AS DOUBLE)
            ELSE CAST(3 AS DOUBLE)
        END AS classe_temperatura_semana
    FROM semana_test
""")

amanha_train_cls.createOrReplaceTempView("amanha_train_cls")
amanha_test_cls.createOrReplaceTempView("amanha_test_cls")
semana_train_cls.createOrReplaceTempView("semana_train_cls")
semana_test_cls.createOrReplaceTempView("semana_test_cls")

Cortes para temperatura de amanhã:
Q1: 18.99
Q2: 21.89
Q3: 24.18

Cortes para temperatura média dos próximos 7 dias:
Q1: 19.08
Q2: 21.90
Q3: 24.03


## 8. Distribuição das classes

Antes de treinar uma rede neural de classificação, é importante verificar a distribuição das classes.

In [103]:
spark.sql("""
    SELECT
        classe_temperatura_amanha,
        CASE
            WHEN classe_temperatura_amanha = 0 THEN 'mais_fria'
            WHEN classe_temperatura_amanha = 1 THEN 'amena_baixa'
            WHEN classe_temperatura_amanha = 2 THEN 'amena_alta'
            WHEN classe_temperatura_amanha = 3 THEN 'mais_quente'
        END AS categoria,
        COUNT(*) AS total_registros
    FROM amanha_train_cls
    GROUP BY classe_temperatura_amanha
    ORDER BY classe_temperatura_amanha
""").show(truncate=False)

spark.sql("""
    SELECT
        classe_temperatura_semana,
        CASE
            WHEN classe_temperatura_semana = 0 THEN 'mais_fria'
            WHEN classe_temperatura_semana = 1 THEN 'amena_baixa'
            WHEN classe_temperatura_semana = 2 THEN 'amena_alta'
            WHEN classe_temperatura_semana = 3 THEN 'mais_quente'
        END AS categoria,
        COUNT(*) AS total_registros
    FROM semana_train_cls
    GROUP BY classe_temperatura_semana
    ORDER BY classe_temperatura_semana
""").show(truncate=False)

+-------------------------+-----------+---------------+
|classe_temperatura_amanha|categoria  |total_registros|
+-------------------------+-----------+---------------+
|0.0                      |mais_fria  |29511          |
|1.0                      |amena_baixa|29987          |
|2.0                      |amena_alta |29826          |
|3.0                      |mais_quente|30908          |
+-------------------------+-----------+---------------+

+-------------------------+-----------+---------------+
|classe_temperatura_semana|categoria  |total_registros|
+-------------------------+-----------+---------------+
|0.0                      |mais_fria  |29621          |
|1.0                      |amena_baixa|30004          |
|2.0                      |amena_alta |29597          |
|3.0                      |mais_quente|31010          |
+-------------------------+-----------+---------------+



Para manter o notebook coerente com Spark, os gráficos serão criados a partir de listas obtidas com `collect()` em resultados pequenos e já agregados.

In [104]:
def spark_df_para_dicts(df):
    return [row.asDict() for row in df.collect()]

### Gráfico: distribuição das classes no treino

In [105]:
dist_amanha = spark.sql("""
    SELECT
        CASE
            WHEN classe_temperatura_amanha = 0 THEN 'mais_fria'
            WHEN classe_temperatura_amanha = 1 THEN 'amena_baixa'
            WHEN classe_temperatura_amanha = 2 THEN 'amena_alta'
            WHEN classe_temperatura_amanha = 3 THEN 'mais_quente'
        END AS categoria,
        COUNT(*) AS total_registros
    FROM amanha_train_cls
    GROUP BY classe_temperatura_amanha
    ORDER BY classe_temperatura_amanha
""")

fig = px.bar(
    spark_df_para_dicts(dist_amanha),
    x="categoria",
    y="total_registros",
    text="total_registros",
    title="Distribuição das classes térmicas — treino amanhã",
    labels={
        "categoria": "Categoria térmica",
        "total_registros": "Total de registros"
    },
    category_orders={
        "categoria": ["mais_fria", "amena_baixa", "amena_alta", "mais_quente"]
    }
)

fig.update_traces(textposition="outside")
fig.show()

In [106]:
dist_semana = spark.sql("""
    SELECT
        CASE
            WHEN classe_temperatura_semana = 0 THEN 'mais_fria'
            WHEN classe_temperatura_semana = 1 THEN 'amena_baixa'
            WHEN classe_temperatura_semana = 2 THEN 'amena_alta'
            WHEN classe_temperatura_semana = 3 THEN 'mais_quente'
        END AS categoria,
        COUNT(*) AS total_registros
    FROM semana_train_cls
    GROUP BY classe_temperatura_semana
    ORDER BY classe_temperatura_semana
""")

fig = px.bar(
    spark_df_para_dicts(dist_semana),
    x="categoria",
    y="total_registros",
    text="total_registros",
    title="Distribuição das classes térmicas — treino próximos 7 dias",
    labels={
        "categoria": "Categoria térmica",
        "total_registros": "Total de registros"
    },
    category_orders={
        "categoria": ["mais_fria", "amena_baixa", "amena_alta", "mais_quente"]
    }
)

fig.update_traces(textposition="outside")
fig.show()

## 9. Definição dos alvos e das features

As features serão identificadas automaticamente, removendo apenas colunas de identificação, alvos numéricos originais e alvos categóricos.

In [107]:
coluna_alvo_amanha_original = "temperatura_amanha"
coluna_alvo_semana_original = "temperatura_media_proximos_7_dias"

coluna_label_amanha = "classe_temperatura_amanha"
coluna_label_semana = "classe_temperatura_semana"

colunas_identificacao = ["station", "station_code", "data_formatada"]

features_amanha = [
    c for c in amanha_train_cls.columns
    if c not in colunas_identificacao + [coluna_alvo_amanha_original, coluna_label_amanha]
]

features_semana = [
    c for c in semana_train_cls.columns
    if c not in colunas_identificacao + [coluna_alvo_semana_original, coluna_label_semana]
]

print(f"Features amanhã ({len(features_amanha)}):")
print(features_amanha)

print(f"\nFeatures semana ({len(features_semana)}):")
print(features_semana)

print("\nClasses presentes no treino amanhã:")
amanha_train_cls.select(coluna_label_amanha).distinct().orderBy(coluna_label_amanha).show()

print("Classes presentes no treino semana:")
semana_train_cls.select(coluna_label_semana).distinct().orderBy(coluna_label_semana).show()

Features amanhã (26):
['ano_imputado', 'mes_sin_imputado', 'mes_cos_imputado', 'latitude_imputado', 'longitude_imputado', 'altitude_imputado', 'temp_media_dia_imputado', 'temp_min_dia_imputado', 'temp_max_dia_imputado', 'temp_orvalho_media_dia_imputado', 'umidade_media_dia_imputado', 'umidade_min_dia_imputado', 'umidade_max_dia_imputado', 'pressao_media_dia_imputado', 'precipitacao_total_dia_imputado', 'radiacao_media_dia_imputado', 'vento_medio_dia_imputado', 'rajada_max_dia_imputado', 'temp_media_ontem_imputado', 'temp_media_ultimos_3_dias_imputado', 'temp_media_ultimos_7_dias_imputado', 'umidade_media_ultimos_7_dias_imputado', 'precipitacao_ultimos_7_dias_imputado', 'macro_regiao_sp_idx', 'tipo_area_idx', 'faixa_altitude_idx']

Features semana (26):
['ano_imputado', 'mes_sin_imputado', 'mes_cos_imputado', 'latitude_imputado', 'longitude_imputado', 'altitude_imputado', 'temp_media_dia_imputado', 'temp_min_dia_imputado', 'temp_max_dia_imputado', 'temp_orvalho_media_dia_imputado', 'umi

Redes neurais são sensíveis à escala das variáveis.

Por isso, antes de treinar a rede neural, as features serão montadas com `VectorAssembler` e padronizadas com `StandardScaler`.

## 11. Treinamento da rede neural para faixa térmica de amanhã

A arquitetura usada será:

- camada de entrada: número de features;
- primeira camada oculta: 32 neurônios;
- segunda camada oculta: 16 neurônios;
- camada de saída: 4 classes.

In [108]:
num_features_amanha = len(features_amanha)
num_classes = 4

layers_amanha = [num_features_amanha, 32, 16, num_classes]

print("Número de features amanhã:", num_features_amanha)
print("Arquitetura da rede amanhã:", layers_amanha)

assembler_amanha = VectorAssembler(
    inputCols=features_amanha,
    outputCol="features_brutas",
    handleInvalid="skip"
)

scaler_amanha = StandardScaler(
    inputCol="features_brutas",
    outputCol="features",
    withMean=True,
    withStd=True
)

mlp_amanha = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=coluna_label_amanha,
    predictionCol="prediction",
    layers=layers_amanha,
    maxIter=500,
    blockSize=64,
    stepSize=0.03,
    seed=42
)

pipeline_amanha = Pipeline(stages=[
    assembler_amanha,
    scaler_amanha,
    mlp_amanha
])

modelo_mlp_amanha = pipeline_amanha.fit(amanha_train_cls)

pred_amanha = modelo_mlp_amanha.transform(amanha_test_cls)
pred_amanha.createOrReplaceTempView("pred_amanha")

print("Rede neural para faixa térmica de amanhã treinada com sucesso.")

Número de features amanhã: 26
Arquitetura da rede amanhã: [26, 32, 16, 4]
Rede neural para faixa térmica de amanhã treinada com sucesso.


In [109]:
spark.sql("""
    SELECT
        classe_temperatura_amanha,
        COUNT(*) AS total
    FROM amanha_train_cls
    GROUP BY classe_temperatura_amanha
    ORDER BY classe_temperatura_amanha
""").show()

+-------------------------+-----+
|classe_temperatura_amanha|total|
+-------------------------+-----+
|                      0.0|29511|
|                      1.0|29987|
|                      2.0|29826|
|                      3.0|30908|
+-------------------------+-----+



In [110]:
spark.sql("""
    SELECT
        prediction,
        COUNT(*) AS total
    FROM pred_amanha
    GROUP BY prediction
    ORDER BY prediction
""").show()

+----------+-----+
|prediction|total|
+----------+-----+
|       0.0| 7559|
|       1.0|11345|
|       2.0|11600|
|       3.0|15436|
+----------+-----+



## 12. Previsões do modelo de amanhã

Após o treinamento, o modelo é aplicado à base de teste.

Nesta etapa, também são criadas colunas interpretáveis para a classe real e a classe prevista.

In [111]:
pred_amanha = spark.sql("""
    SELECT *,
        CASE
            WHEN classe_temperatura_amanha = 0 THEN 'mais_fria'
            WHEN classe_temperatura_amanha = 1 THEN 'amena_baixa'
            WHEN classe_temperatura_amanha = 2 THEN 'amena_alta'
            WHEN classe_temperatura_amanha = 3 THEN 'mais_quente'
        END AS categoria_real,

        CASE
            WHEN prediction = 0 THEN 'mais_fria'
            WHEN prediction = 1 THEN 'amena_baixa'
            WHEN prediction = 2 THEN 'amena_alta'
            WHEN prediction = 3 THEN 'mais_quente'
        END AS categoria_prevista,

        CASE
            WHEN classe_temperatura_amanha = prediction THEN 1
            ELSE 0
        END AS acertou
    FROM pred_amanha
""")

pred_amanha.createOrReplaceTempView("pred_amanha")

spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        ROUND(temperatura_amanha, 2) AS temperatura_real,
        classe_temperatura_amanha,
        prediction,
        categoria_real,
        categoria_prevista,
        acertou
    FROM pred_amanha
    LIMIT 20
""").show(truncate=False)

+--------+------------+--------------+----------------+-------------------------+----------+--------------+------------------+-------+
|station |station_code|data_formatada|temperatura_real|classe_temperatura_amanha|prediction|categoria_real|categoria_prevista|acertou|
+--------+------------+--------------+----------------+-------------------------+----------+--------------+------------------+-------+
|SOROCABA|A713        |2018-01-01    |22.82           |2.0                      |2.0       |amena_alta    |amena_alta        |1      |
|SOROCABA|A713        |2018-01-02    |22.72           |2.0                      |2.0       |amena_alta    |amena_alta        |1      |
|SOROCABA|A713        |2018-01-03    |23.15           |2.0                      |2.0       |amena_alta    |amena_alta        |1      |
|SOROCABA|A713        |2018-01-04    |23.26           |2.0                      |2.0       |amena_alta    |amena_alta        |1      |
|SOROCABA|A713        |2018-01-05    |22.95           |

In [112]:
spark.sql("""
    SELECT
        categoria_real,
        categoria_prevista,
        COUNT(*) AS total
    FROM pred_amanha
    GROUP BY categoria_real, categoria_prevista
    ORDER BY categoria_real, categoria_prevista
""").show(100, truncate=False)

+--------------+------------------+-----+
|categoria_real|categoria_prevista|total|
+--------------+------------------+-----+
|amena_alta    |amena_alta        |6932 |
|amena_alta    |amena_baixa       |2274 |
|amena_alta    |mais_fria         |76   |
|amena_alta    |mais_quente       |2653 |
|amena_baixa   |amena_alta        |1995 |
|amena_baixa   |amena_baixa       |7094 |
|amena_baixa   |mais_fria         |1124 |
|amena_baixa   |mais_quente       |317  |
|mais_fria     |amena_alta        |104  |
|mais_fria     |amena_baixa       |1727 |
|mais_fria     |mais_fria         |6338 |
|mais_fria     |mais_quente       |45   |
|mais_quente   |amena_alta        |2569 |
|mais_quente   |amena_baixa       |250  |
|mais_quente   |mais_fria         |21   |
|mais_quente   |mais_quente       |12421|
+--------------+------------------+-----+



In [113]:
spark.sql("""
    SELECT
        prediction,
        categoria_prevista,
        COUNT(*) AS total
    FROM pred_amanha
    GROUP BY prediction, categoria_prevista
    ORDER BY prediction
""").show(truncate=False)

+----------+------------------+-----+
|prediction|categoria_prevista|total|
+----------+------------------+-----+
|0.0       |mais_fria         |7559 |
|1.0       |amena_baixa       |11345|
|2.0       |amena_alta        |11600|
|3.0       |mais_quente       |15436|
+----------+------------------+-----+



## 13. Função de avaliação de classificação

Serão usadas as seguintes métricas:

- **Accuracy:** percentual geral de acertos.
- **F1-score:** métrica que combina precisão e revocação.
- **Weighted Precision:** precisão ponderada pelo tamanho das classes.
- **Weighted Recall:** revocação ponderada pelo tamanho das classes.

In [114]:
def avaliar_classificacao(predicoes, coluna_label, nome_modelo):
    avaliador_accuracy = MulticlassClassificationEvaluator(
        labelCol=coluna_label,
        predictionCol="prediction",
        metricName="accuracy"
    )

    avaliador_f1 = MulticlassClassificationEvaluator(
        labelCol=coluna_label,
        predictionCol="prediction",
        metricName="f1"
    )

    avaliador_precision = MulticlassClassificationEvaluator(
        labelCol=coluna_label,
        predictionCol="prediction",
        metricName="weightedPrecision"
    )

    avaliador_recall = MulticlassClassificationEvaluator(
        labelCol=coluna_label,
        predictionCol="prediction",
        metricName="weightedRecall"
    )

    accuracy = avaliador_accuracy.evaluate(predicoes)
    f1 = avaliador_f1.evaluate(predicoes)
    precision = avaliador_precision.evaluate(predicoes)
    recall = avaliador_recall.evaluate(predicoes)

    print(nome_modelo)
    print(f"Accuracy          : {accuracy:.4f}")
    print(f"F1-score          : {f1:.4f}")
    print(f"Weighted Precision: {precision:.4f}")
    print(f"Weighted Recall   : {recall:.4f}")

    return {
        "modelo": nome_modelo,
        "accuracy": float(accuracy),
        "f1": float(f1),
        "weighted_precision": float(precision),
        "weighted_recall": float(recall)
    }

metricas_amanha = avaliar_classificacao(
    pred_amanha,
    coluna_label_amanha,
    "Rede Neural MLlib - Faixa térmica amanhã"
)

Rede Neural MLlib - Faixa térmica amanhã
Accuracy          : 0.7136
F1-score          : 0.7142
Weighted Precision: 0.7158
Weighted Recall   : 0.7136


## 14. Treinamento da rede neural para faixa térmica dos próximos 7 dias

In [115]:
num_features_semana = len(features_semana)
num_classes = 4

layers_semana = [num_features_semana, 32, 16, num_classes]

print("Número de features semana:", num_features_semana)
print("Arquitetura da rede semana:", layers_semana)

assembler_semana = VectorAssembler(
    inputCols=features_semana,
    outputCol="features_brutas",
    handleInvalid="skip"
)

scaler_semana = StandardScaler(
    inputCol="features_brutas",
    outputCol="features",
    withMean=True,
    withStd=True
)

mlp_semana = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=coluna_label_semana,
    predictionCol="prediction",
    layers=layers_semana,
    maxIter=500,
    blockSize=64,
    stepSize=0.03,
    seed=42
)

pipeline_semana = Pipeline(stages=[
    assembler_semana,
    scaler_semana,
    mlp_semana
])

modelo_mlp_semana = pipeline_semana.fit(semana_train_cls)

pred_semana = modelo_mlp_semana.transform(semana_test_cls)
pred_semana.createOrReplaceTempView("pred_semana")

print("Rede neural para faixa térmica média dos próximos 7 dias treinada com sucesso.")

Número de features semana: 26
Arquitetura da rede semana: [26, 32, 16, 4]
Rede neural para faixa térmica média dos próximos 7 dias treinada com sucesso.


In [116]:
spark.sql("""
    SELECT
        classe_temperatura_semana,
        COUNT(*) AS total
    FROM semana_train_cls
    GROUP BY classe_temperatura_semana
    ORDER BY classe_temperatura_semana
""").show()

+-------------------------+-----+
|classe_temperatura_semana|total|
+-------------------------+-----+
|                      0.0|29621|
|                      1.0|30004|
|                      2.0|29597|
|                      3.0|31010|
+-------------------------+-----+



In [117]:
spark.sql("""
    SELECT
        prediction,
        COUNT(*) AS total
    FROM pred_semana
    GROUP BY prediction
    ORDER BY prediction
""").show()

+----------+-----+
|prediction|total|
+----------+-----+
|       0.0| 7773|
|       1.0|11605|
|       2.0|11521|
|       3.0|15041|
+----------+-----+



## 15. Previsões do modelo dos próximos 7 dias

In [118]:
pred_semana = spark.sql("""
    SELECT *,
        CASE
            WHEN classe_temperatura_semana = 0 THEN 'mais_fria'
            WHEN classe_temperatura_semana = 1 THEN 'amena_baixa'
            WHEN classe_temperatura_semana = 2 THEN 'amena_alta'
            WHEN classe_temperatura_semana = 3 THEN 'mais_quente'
        END AS categoria_real,

        CASE
            WHEN prediction = 0 THEN 'mais_fria'
            WHEN prediction = 1 THEN 'amena_baixa'
            WHEN prediction = 2 THEN 'amena_alta'
            WHEN prediction = 3 THEN 'mais_quente'
        END AS categoria_prevista,

        CASE
            WHEN classe_temperatura_semana = prediction THEN 1
            ELSE 0
        END AS acertou
    FROM pred_semana
""")

pred_semana.createOrReplaceTempView("pred_semana")

spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        ROUND(temperatura_media_proximos_7_dias, 2) AS temperatura_real,
        classe_temperatura_semana,
        prediction,
        categoria_real,
        categoria_prevista,
        acertou
    FROM pred_semana
    LIMIT 20
""").show(truncate=False)

+--------+------------+--------------+----------------+-------------------------+----------+--------------+------------------+-------+
|station |station_code|data_formatada|temperatura_real|classe_temperatura_semana|prediction|categoria_real|categoria_prevista|acertou|
+--------+------------+--------------+----------------+-------------------------+----------+--------------+------------------+-------+
|SOROCABA|A713        |2018-01-01    |22.44           |2.0                      |2.0       |amena_alta    |amena_alta        |1      |
|SOROCABA|A713        |2018-01-02    |22.14           |2.0                      |2.0       |amena_alta    |amena_alta        |1      |
|SOROCABA|A713        |2018-01-03    |22.28           |2.0                      |2.0       |amena_alta    |amena_alta        |1      |
|SOROCABA|A713        |2018-01-04    |22.19           |2.0                      |2.0       |amena_alta    |amena_alta        |1      |
|SOROCABA|A713        |2018-01-05    |21.8            |

In [119]:
spark.sql("""
    SELECT
        categoria_real,
        categoria_prevista,
        COUNT(*) AS total
    FROM pred_semana
    GROUP BY categoria_real, categoria_prevista
    ORDER BY categoria_real, categoria_prevista
""").show(100, truncate=False)

+--------------+------------------+-----+
|categoria_real|categoria_prevista|total|
+--------------+------------------+-----+
|amena_alta    |amena_alta        |5482 |
|amena_alta    |amena_baixa       |2903 |
|amena_alta    |mais_fria         |216  |
|amena_alta    |mais_quente       |3166 |
|amena_baixa   |amena_alta        |2386 |
|amena_baixa   |amena_baixa       |5865 |
|amena_baixa   |mais_fria         |2075 |
|amena_baixa   |mais_quente       |397  |
|mais_fria     |amena_alta        |133  |
|mais_fria     |amena_baixa       |2277 |
|mais_fria     |mais_fria         |5453 |
|mais_fria     |mais_quente       |16   |
|mais_quente   |amena_alta        |3520 |
|mais_quente   |amena_baixa       |560  |
|mais_quente   |mais_fria         |29   |
|mais_quente   |mais_quente       |11462|
+--------------+------------------+-----+



In [120]:
spark.sql("""
    SELECT
        prediction,
        categoria_prevista,
        COUNT(*) AS total
    FROM pred_semana
    GROUP BY prediction, categoria_prevista
    ORDER BY prediction
""").show(truncate=False)

+----------+------------------+-----+
|prediction|categoria_prevista|total|
+----------+------------------+-----+
|0.0       |mais_fria         |7773 |
|1.0       |amena_baixa       |11605|
|2.0       |amena_alta        |11521|
|3.0       |mais_quente       |15041|
+----------+------------------+-----+



In [121]:
metricas_semana = avaliar_classificacao(
    pred_semana,
    coluna_label_semana,
    "Rede Neural MLlib - Faixa térmica próximos 7 dias"
)

Rede Neural MLlib - Faixa térmica próximos 7 dias
Accuracy          : 0.6152
F1-score          : 0.6165
Weighted Precision: 0.6184
Weighted Recall   : 0.6152


## 16. Comparação final das métricas

In [122]:
metricas_mlp = spark.createDataFrame([metricas_amanha, metricas_semana])
metricas_mlp.createOrReplaceTempView("metricas_mlp")

spark.sql("""
    SELECT
        modelo,
        ROUND(accuracy, 4) AS accuracy,
        ROUND(f1, 4) AS f1,
        ROUND(weighted_precision, 4) AS weighted_precision,
        ROUND(weighted_recall, 4) AS weighted_recall
    FROM metricas_mlp
""").show(truncate=False)

+-------------------------------------------------+--------+------+------------------+---------------+
|modelo                                           |accuracy|f1    |weighted_precision|weighted_recall|
+-------------------------------------------------+--------+------+------------------+---------------+
|Rede Neural MLlib - Faixa térmica amanhã         |0.7136  |0.7142|0.7158            |0.7136         |
|Rede Neural MLlib - Faixa térmica próximos 7 dias|0.6152  |0.6165|0.6184            |0.6152         |
+-------------------------------------------------+--------+------+------------------+---------------+



### Gráfico: comparação da acurácia

In [123]:
metricas_accuracy_plot_df = spark.sql("""
    SELECT modelo, accuracy
    FROM metricas_mlp
    ORDER BY modelo
""")

fig = px.bar(
    spark_df_para_dicts(metricas_accuracy_plot_df),
    x="modelo",
    y="accuracy",
    text="accuracy",
    title="Comparação da acurácia entre os modelos de rede neural",
    labels={"modelo": "Modelo", "accuracy": "Acurácia"}
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

In [124]:
metricas_f1_plot_df = spark.sql("""
    SELECT modelo, f1
    FROM metricas_mlp
    ORDER BY modelo
""")

fig = px.bar(
    spark_df_para_dicts(metricas_f1_plot_df),
    x="modelo",
    y="f1",
    text="f1",
    title="Comparação do F1-score entre os modelos de rede neural",
    labels={"modelo": "Modelo", "f1": "F1-score"}
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Insight esperado

Se o modelo dos próximos 7 dias apresentar acurácia maior, isso pode indicar que médias semanais geram classes mais estáveis.

## 17. Matriz de confusão

A matriz de confusão mostra onde o modelo acerta e onde ele erra.

Como as classes representam faixas de temperatura, é esperado que as confusões mais comuns ocorram entre categorias próximas.

In [125]:
matriz_confusao_amanha = spark.sql("""
    SELECT 
        categoria_real,
        categoria_prevista,
        COUNT(*) AS total
    FROM pred_amanha
    GROUP BY categoria_real, categoria_prevista
    ORDER BY categoria_real, categoria_prevista
""")

matriz_confusao_semana = spark.sql("""
    SELECT 
        categoria_real,
        categoria_prevista,
        COUNT(*) AS total
    FROM pred_semana
    GROUP BY categoria_real, categoria_prevista
    ORDER BY categoria_real, categoria_prevista
""")

matriz_confusao_amanha.createOrReplaceTempView("matriz_confusao_amanha")
matriz_confusao_semana.createOrReplaceTempView("matriz_confusao_semana")

matriz_confusao_amanha.show(100, truncate=False)
matriz_confusao_semana.show(100, truncate=False)

+--------------+------------------+-----+
|categoria_real|categoria_prevista|total|
+--------------+------------------+-----+
|amena_alta    |amena_alta        |6932 |
|amena_alta    |amena_baixa       |2274 |
|amena_alta    |mais_fria         |76   |
|amena_alta    |mais_quente       |2653 |
|amena_baixa   |amena_alta        |1995 |
|amena_baixa   |amena_baixa       |7094 |
|amena_baixa   |mais_fria         |1124 |
|amena_baixa   |mais_quente       |317  |
|mais_fria     |amena_alta        |104  |
|mais_fria     |amena_baixa       |1727 |
|mais_fria     |mais_fria         |6338 |
|mais_fria     |mais_quente       |45   |
|mais_quente   |amena_alta        |2569 |
|mais_quente   |amena_baixa       |250  |
|mais_quente   |mais_fria         |21   |
|mais_quente   |mais_quente       |12421|
+--------------+------------------+-----+

+--------------+------------------+-----+
|categoria_real|categoria_prevista|total|
+--------------+------------------+-----+
|amena_alta    |amena_alta       

### Gráfico: matriz de confusão — amanhã

In [126]:
fig = px.density_heatmap(
    spark_df_para_dicts(matriz_confusao_amanha),
    x="categoria_prevista",
    y="categoria_real",
    z="total",
    text_auto=True,
    title="Matriz de confusão — faixa térmica amanhã",
    labels={
        "categoria_prevista": "Categoria prevista",
        "categoria_real": "Categoria real",
        "total": "Total"
    },
    category_orders={
        "categoria_real": ["mais_fria", "amena_baixa", "amena_alta", "mais_quente"],
        "categoria_prevista": ["mais_fria", "amena_baixa", "amena_alta", "mais_quente"]
    }
)

fig.show()

### Gráfico: matriz de confusão — próximos 7 dias

In [127]:
fig = px.density_heatmap(
    spark_df_para_dicts(matriz_confusao_semana),
    x="categoria_prevista",
    y="categoria_real",
    z="total",
    text_auto=True,
    title="Matriz de confusão — faixa térmica próximos 7 dias",
    labels={
        "categoria_prevista": "Categoria prevista",
        "categoria_real": "Categoria real",
        "total": "Total"
    },
    category_orders={
        "categoria_real": ["mais_fria", "amena_baixa", "amena_alta", "mais_quente"],
        "categoria_prevista": ["mais_fria", "amena_baixa", "amena_alta", "mais_quente"]
    }
)

fig.show()

## 18. Recuperação das informações geográficas interpretáveis

Para criar gráficos e insights por região, vamos recuperar da base completa algumas colunas geográficas:

- `tipo_area`;
- `macro_regiao_sp`;
- `faixa_altitude`;
- `altitude`;
- `latitude`;
- `longitude`.

In [128]:
contexto_amanha = spark.sql("""
    SELECT DISTINCT
        station_code,
        data_formatada,
        altitude_imputado AS altitude,
        latitude_imputado AS latitude,
        longitude_imputado AS longitude
    FROM dataset_amanha_completo
""")

contexto_semana = spark.sql("""
    SELECT DISTINCT
        station_code,
        data_formatada,
        altitude_imputado AS altitude,
        latitude_imputado AS latitude,
        longitude_imputado AS longitude
    FROM dataset_semana_completo
""")

contexto_amanha.createOrReplaceTempView("contexto_amanha")
contexto_semana.createOrReplaceTempView("contexto_semana")

pred_amanha_ctx = spark.sql("""
    SELECT 
        p.*,
        c.altitude,
        c.latitude,
        c.longitude
    FROM pred_amanha p
    LEFT JOIN contexto_amanha c
        ON p.station_code = c.station_code
       AND p.data_formatada = c.data_formatada
""")

pred_semana_ctx = spark.sql("""
    SELECT 
        p.*,
        c.altitude,
        c.latitude,
        c.longitude
    FROM pred_semana p
    LEFT JOIN contexto_semana c
        ON p.station_code = c.station_code
       AND p.data_formatada = c.data_formatada
""")

pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")
pred_semana_ctx.createOrReplaceTempView("pred_semana_ctx")

In [129]:
spark.sql("""
    SELECT
        station,
        data_formatada,
        tipo_area_idx,
        macro_regiao_sp_idx,
        faixa_altitude_idx,
        categoria_real,
        categoria_prevista,
        acertou
    FROM pred_amanha_ctx
    LIMIT 20
""").show(truncate=False)

+--------+--------------+-------------+-------------------+------------------+--------------+------------------+-------+
|station |data_formatada|tipo_area_idx|macro_regiao_sp_idx|faixa_altitude_idx|categoria_real|categoria_prevista|acertou|
+--------+--------------+-------------+-------------------+------------------+--------------+------------------+-------+
|SOROCABA|2018-01-01    |0.0          |1.0                |0.0               |amena_alta    |amena_alta        |1      |
|SOROCABA|2018-01-02    |0.0          |1.0                |0.0               |amena_alta    |amena_alta        |1      |
|SOROCABA|2018-01-03    |0.0          |1.0                |0.0               |amena_alta    |amena_alta        |1      |
|SOROCABA|2018-01-04    |0.0          |1.0                |0.0               |amena_alta    |amena_alta        |1      |
|SOROCABA|2018-01-05    |0.0          |1.0                |0.0               |amena_alta    |amena_alta        |1      |
|SOROCABA|2018-01-06    |0.0    

## 19. Acurácia por tipo de área

Esta análise permite observar diferenças entre áreas urbanas/metropolitanas, litorâneas, serranas e interiores.

In [130]:
acuracia_tipo_area_amanha = spark.sql("""
    SELECT
        tipo_area_idx,
        COUNT(*) AS total_registros,
        ROUND(AVG(acertou), 4) AS accuracy
    FROM pred_amanha_ctx
    WHERE tipo_area_idx IS NOT NULL
    GROUP BY tipo_area_idx
    ORDER BY tipo_area_idx
""")

acuracia_tipo_area_semana = spark.sql("""
    SELECT
        tipo_area_idx,
        COUNT(*) AS total_registros,
        ROUND(AVG(acertou), 4) AS accuracy
    FROM pred_semana_ctx
    WHERE tipo_area_idx IS NOT NULL
    GROUP BY tipo_area_idx
    ORDER BY tipo_area_idx
""")

acuracia_tipo_area_amanha.createOrReplaceTempView("acuracia_tipo_area_amanha")
acuracia_tipo_area_semana.createOrReplaceTempView("acuracia_tipo_area_semana")

acuracia_tipo_area_amanha.show(truncate=False)
acuracia_tipo_area_semana.show(truncate=False)

+-------------+---------------+--------+
|tipo_area_idx|total_registros|accuracy|
+-------------+---------------+--------+
|0.0          |30049          |0.7191  |
|1.0          |13639          |0.6977  |
|2.0          |2252           |0.738   |
+-------------+---------------+--------+

+-------------+---------------+--------+
|tipo_area_idx|total_registros|accuracy|
+-------------+---------------+--------+
|0.0          |30049          |0.6142  |
|1.0          |13639          |0.6077  |
|2.0          |2252           |0.6732  |
+-------------+---------------+--------+



### Gráfico: acurácia por tipo de área — amanhã

In [131]:
fig = px.bar(
    spark_df_para_dicts(acuracia_tipo_area_amanha),
    x="tipo_area_idx",
    y="accuracy",
    text="accuracy",
    title="Acurácia por tipo de área — faixa térmica amanhã",
    labels={
        "tipo_area_idx": "Tipo de área",
        "accuracy": "Acurácia"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Gráfico: acurácia por tipo de área — próximos 7 dias

In [132]:
fig = px.bar(
    spark_df_para_dicts(acuracia_tipo_area_semana),
    x="tipo_area_idx",
    y="accuracy",
    text="accuracy",
    title="Acurácia por tipo de área — faixa térmica próximos 7 dias",
    labels={
        "tipo_area_idx": "Tipo de área",
        "accuracy": "Acurácia"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Insight esperado

Se a acurácia for menor em `serra_altitude`, isso pode indicar que a dinâmica térmica dessas áreas é mais específica por causa do relevo e da altitude.

Se a acurácia for menor em `litoral`, pode haver influência de umidade, brisa marítima e menor amplitude térmica.

Se a acurácia for menor em `urbano_metropolitano`, pode ser um indício de que efeitos locais, como concentração urbana e ilha de calor, são relevantes para a classificação térmica.

## 20. Distribuição das categorias reais por tipo de área

In [133]:
classe_tipo_area_amanha = spark.sql("""
    SELECT
        tipo_area_idx,
        categoria_real,
        COUNT(*) AS total_registros
    FROM pred_amanha_ctx
    WHERE tipo_area_idx IS NOT NULL
      AND categoria_real IS NOT NULL
    GROUP BY tipo_area_idx, categoria_real
    ORDER BY tipo_area_idx, categoria_real
""")

classe_tipo_area_amanha.createOrReplaceTempView("classe_tipo_area_amanha")

classe_tipo_area_amanha.show(100, truncate=False)

+-------------+--------------+---------------+
|tipo_area_idx|categoria_real|total_registros|
+-------------+--------------+---------------+
|0.0          |amena_alta    |8071           |
|0.0          |amena_baixa   |6157           |
|0.0          |mais_fria     |4078           |
|0.0          |mais_quente   |11743          |
|1.0          |amena_alta    |3289           |
|1.0          |amena_baixa   |3966           |
|1.0          |mais_fria     |4016           |
|1.0          |mais_quente   |2368           |
|2.0          |amena_alta    |575            |
|2.0          |amena_baixa   |407            |
|2.0          |mais_fria     |120            |
|2.0          |mais_quente   |1150           |
+-------------+--------------+---------------+



In [134]:
fig = px.bar(
    spark_df_para_dicts(classe_tipo_area_amanha),
    x="tipo_area_idx",
    y="total_registros",
    color="categoria_real",
    barmode="group",
    title="Distribuição das categorias reais por tipo de área — amanhã",
    labels={
        "tipo_area_idx": "Tipo de área",
        "total_registros": "Total de registros",
        "categoria_real": "Categoria real"
    },
    category_orders={
        "categoria_real": ["mais_fria", "amena_baixa", "amena_alta", "mais_quente"]
    }
)

fig.show()

In [135]:
classe_tipo_area_semana = spark.sql("""
    SELECT
        tipo_area_idx,
        categoria_real,
        COUNT(*) AS total_registros
    FROM pred_semana_ctx
    WHERE tipo_area_idx IS NOT NULL
      AND categoria_real IS NOT NULL
    GROUP BY tipo_area_idx, categoria_real
    ORDER BY tipo_area_idx, categoria_real
""")

classe_tipo_area_semana.createOrReplaceTempView("classe_tipo_area_semana")

classe_tipo_area_semana.show(100, truncate=False)

+-------------+--------------+---------------+
|tipo_area_idx|categoria_real|total_registros|
+-------------+--------------+---------------+
|0.0          |amena_alta    |7630           |
|0.0          |amena_baixa   |6215           |
|0.0          |mais_fria     |3858           |
|0.0          |mais_quente   |12346          |
|1.0          |amena_alta    |3569           |
|1.0          |amena_baixa   |4112           |
|1.0          |mais_fria     |3948           |
|1.0          |mais_quente   |2010           |
|2.0          |amena_alta    |568            |
|2.0          |amena_baixa   |396            |
|2.0          |mais_fria     |73             |
|2.0          |mais_quente   |1215           |
+-------------+--------------+---------------+



In [136]:
fig = px.bar(
    spark_df_para_dicts(classe_tipo_area_semana),
    x="tipo_area_idx",
    y="total_registros",
    color="categoria_real",
    barmode="group",
    title="Distribuição das categorias reais por tipo de área — próximos 7 dias",
    labels={
        "tipo_area_idx": "Tipo de área",
        "total_registros": "Total de registros",
        "categoria_real": "Categoria real"
    },
    category_orders={
        "categoria_real": ["mais_fria", "amena_baixa", "amena_alta", "mais_quente"]
    }
)

fig.show()

### Insight esperado

A distribuição das categorias pode revelar padrões climáticos coerentes:

- áreas de `serra_altitude` tendem a concentrar mais registros mais_frias ou amenos;
- áreas `urbano_metropolitano` podem ter maior presença de categorias quentes;
- o `litoral` pode apresentar comportamento mais ameno ou estável;
- o `interior` pode apresentar maior amplitude térmica em determinadas épocas do ano.

## 21. Acurácia por mês e tipo de área

A análise mensal mostra se o modelo erra mais em determinados períodos do ano.

In [137]:
pred_amanha_ctx = spark.sql("""
    SELECT 
        *,
        MONTH(data_formatada) AS mes
    FROM pred_amanha_ctx
""")

pred_amanha_ctx.createOrReplaceTempView("pred_amanha_ctx")

acuracia_mes_tipo_area_amanha = spark.sql("""
    SELECT
        mes,
        tipo_area_idx,
        COUNT(*) AS total_registros,
        ROUND(AVG(acertou), 4) AS accuracy
    FROM pred_amanha_ctx
    WHERE mes IS NOT NULL
      AND tipo_area_idx IS NOT NULL
    GROUP BY mes, tipo_area_idx
    ORDER BY mes, tipo_area_idx
""")

acuracia_mes_tipo_area_amanha.createOrReplaceTempView("acuracia_mes_tipo_area_amanha")

acuracia_mes_tipo_area_amanha.show(100, truncate=False)

+---+-------------+---------------+--------+
|mes|tipo_area_idx|total_registros|accuracy|
+---+-------------+---------------+--------+
|1  |0.0          |2961           |0.7842  |
|1  |1.0          |1416           |0.6667  |
|1  |2.0          |218            |0.9404  |
|2  |0.0          |2669           |0.7115  |
|2  |1.0          |1269           |0.6367  |
|2  |2.0          |186            |0.8656  |
|3  |0.0          |3015           |0.7715  |
|3  |1.0          |1376           |0.7188  |
|3  |2.0          |216            |0.9074  |
|4  |0.0          |2897           |0.7197  |
|4  |1.0          |1284           |0.7274  |
|4  |2.0          |206            |0.7476  |
|5  |0.0          |2421           |0.6927  |
|5  |1.0          |1081           |0.7669  |
|5  |2.0          |185            |0.6432  |
|6  |0.0          |2301           |0.678   |
|6  |1.0          |1043           |0.745   |
|6  |2.0          |170            |0.6412  |
|7  |0.0          |2408           |0.7338  |
|7  |1.0  

In [138]:
fig = px.line(
    spark_df_para_dicts(acuracia_mes_tipo_area_amanha),
    x="mes",
    y="accuracy",
    color="tipo_area_idx",
    markers=True,
    title="Acurácia por mês e tipo de área — faixa térmica amanhã",
    labels={
        "mes": "Mês",
        "accuracy": "Acurácia",
        "tipo_area_idx": "Tipo de área"
    }
)

fig.show()

In [139]:
pred_semana_ctx = spark.sql("""
    SELECT 
        *,
        MONTH(data_formatada) AS mes
    FROM pred_semana_ctx
""")

pred_semana_ctx.createOrReplaceTempView("pred_semana_ctx")

acuracia_mes_tipo_area_semana = spark.sql("""
    SELECT
        mes,
        tipo_area_idx,
        COUNT(*) AS total_registros,
        ROUND(AVG(acertou), 4) AS accuracy
    FROM pred_semana_ctx
    WHERE mes IS NOT NULL
      AND tipo_area_idx IS NOT NULL
    GROUP BY mes, tipo_area_idx
    ORDER BY mes, tipo_area_idx
""")

acuracia_mes_tipo_area_semana.createOrReplaceTempView("acuracia_mes_tipo_area_semana")

acuracia_mes_tipo_area_semana.show(100, truncate=False)

+---+-------------+---------------+--------+
|mes|tipo_area_idx|total_registros|accuracy|
+---+-------------+---------------+--------+
|1  |0.0          |2961           |0.7778  |
|1  |1.0          |1416           |0.5862  |
|1  |2.0          |218            |0.9862  |
|2  |0.0          |2669           |0.6324  |
|2  |1.0          |1269           |0.5461  |
|2  |2.0          |186            |0.9409  |
|3  |0.0          |3015           |0.7376  |
|3  |1.0          |1376           |0.6417  |
|3  |2.0          |216            |0.8565  |
|4  |0.0          |2897           |0.4643  |
|4  |1.0          |1284           |0.5374  |
|4  |2.0          |206            |0.4466  |
|5  |0.0          |2421           |0.5056  |
|5  |1.0          |1081           |0.6549  |
|5  |2.0          |185            |0.4973  |
|6  |0.0          |2301           |0.5259  |
|6  |1.0          |1043           |0.627   |
|6  |2.0          |170            |0.5824  |
|7  |0.0          |2408           |0.5772  |
|7  |1.0  

In [140]:
fig = px.line(
    spark_df_para_dicts(acuracia_mes_tipo_area_semana),
    x="mes",
    y="accuracy",
    color="tipo_area_idx",
    markers=True,
    title="Acurácia por mês e tipo de área — faixa térmica próximos 7 dias",
    labels={
        "mes": "Mês",
        "accuracy": "Acurácia",
        "tipo_area_idx": "Tipo de área"
    }
)

fig.show()

### Insight esperado

Se a acurácia cair em meses de transição, como outono e primavera, isso pode indicar que o modelo tem mais dificuldade em períodos de maior variabilidade atmosférica.

## 22. Acurácia por macro região

In [141]:
acuracia_macro_amanha = spark.sql("""
    SELECT
        macro_regiao_sp_idx,
        COUNT(*) AS total_registros,
        ROUND(AVG(acertou), 4) AS accuracy
    FROM pred_amanha_ctx
    WHERE macro_regiao_sp_idx IS NOT NULL
    GROUP BY macro_regiao_sp_idx
    ORDER BY macro_regiao_sp_idx
""")

acuracia_macro_amanha.createOrReplaceTempView("acuracia_macro_amanha")

acuracia_macro_amanha.show(truncate=False)

+-------------------+---------------+--------+
|macro_regiao_sp_idx|total_registros|accuracy|
+-------------------+---------------+--------+
|0.0                |19263          |0.725   |
|1.0                |13315          |0.7012  |
|2.0                |6964           |0.7298  |
|3.0                |1220           |0.6467  |
|4.0                |5178           |0.6976  |
+-------------------+---------------+--------+



In [142]:
fig = px.bar(
    spark_df_para_dicts(acuracia_macro_amanha),
    x="macro_regiao_sp_idx",
    y="accuracy",
    text="accuracy",
    title="Acurácia por macro região — faixa térmica amanhã",
    labels={
        "macro_regiao_sp_idx": "Macro região",
        "accuracy": "Acurácia"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

In [143]:
acuracia_macro_semana = spark.sql("""
    SELECT
        macro_regiao_sp_idx,
        COUNT(*) AS total_registros,
        ROUND(AVG(acertou), 4) AS accuracy
    FROM pred_semana_ctx
    WHERE macro_regiao_sp_idx IS NOT NULL
    GROUP BY macro_regiao_sp_idx
    ORDER BY macro_regiao_sp_idx
""")

acuracia_macro_semana.createOrReplaceTempView("acuracia_macro_semana")

acuracia_macro_semana.show(truncate=False)

+-------------------+---------------+--------+
|macro_regiao_sp_idx|total_registros|accuracy|
+-------------------+---------------+--------+
|0.0                |19263          |0.6212  |
|1.0                |13315          |0.5848  |
|2.0                |6964           |0.6409  |
|3.0                |1220           |0.618   |
|4.0                |5178           |0.6358  |
+-------------------+---------------+--------+



In [144]:
fig = px.bar(
    spark_df_para_dicts(acuracia_macro_semana),
    x="macro_regiao_sp_idx",
    y="accuracy",
    text="accuracy",
    title="Acurácia por macro região — faixa térmica próximos 7 dias",
    labels={
        "macro_regiao_sp_idx": "Macro região",
        "accuracy": "Acurácia"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

## 23. Relação entre altitude e acerto

A altitude é uma variável importante para temperatura.

Este gráfico ajuda a observar se o modelo tem mais dificuldade em regiões mais altas, como áreas de serra.

Para manter o gráfico leve, a consulta usa uma amostra limitada.

In [145]:
altitude_acerto = spark.sql("""
    SELECT
        altitude,
        tipo_area_idx,
        acertou
    FROM pred_amanha_ctx
    WHERE altitude IS NOT NULL
      AND tipo_area_idx IS NOT NULL
      AND acertou IS NOT NULL
    LIMIT 5000
""")

altitude_acerto.createOrReplaceTempView("altitude_acerto")

altitude_acerto.show(20, truncate=False)

+--------+-------------+-------+
|altitude|tipo_area_idx|acertou|
+--------+-------------+-------+
|1023.05 |1.0          |1      |
|1023.05 |1.0          |0      |
|1023.05 |1.0          |0      |
|1023.05 |1.0          |1      |
|571.96  |0.0          |0      |
|571.96  |0.0          |0      |
|571.96  |0.0          |0      |
|571.96  |0.0          |1      |
|571.96  |0.0          |1      |
|571.96  |0.0          |1      |
|730.7   |1.0          |0      |
|730.7   |1.0          |1      |
|730.7   |1.0          |0      |
|730.7   |1.0          |1      |
|771.0   |1.0          |1      |
|771.0   |1.0          |1      |
|771.0   |1.0          |0      |
|771.0   |1.0          |0      |
|771.0   |1.0          |1      |
|771.0   |1.0          |0      |
+--------+-------------+-------+
only showing top 20 rows



In [146]:
fig = px.scatter(
    spark_df_para_dicts(altitude_acerto),
    x="altitude",
    y="acertou",
    color="tipo_area_idx",
    opacity=0.5,
    title="Relação entre altitude e acerto — faixa térmica amanhã",
    labels={
        "altitude": "Altitude (m)",
        "acertou": "Acertou a classe",
        "tipo_area_idx": "Tipo de área"
    }
)

fig.show()

In [147]:
acuracia_faixa_altitude_amanha = spark.sql("""
    SELECT
        faixa_altitude_idx,
        COUNT(*) AS total_registros,
        ROUND(AVG(acertou), 4) AS accuracy,
        ROUND(AVG(altitude), 2) AS altitude_media
    FROM pred_amanha_ctx
    WHERE faixa_altitude_idx IS NOT NULL
      AND altitude IS NOT NULL
      AND acertou IS NOT NULL
    GROUP BY faixa_altitude_idx
    ORDER BY faixa_altitude_idx
""")

acuracia_faixa_altitude_amanha.createOrReplaceTempView("acuracia_faixa_altitude_amanha")

acuracia_faixa_altitude_amanha.show(truncate=False)

+------------------+---------------+--------+--------------+
|faixa_altitude_idx|total_registros|accuracy|altitude_media|
+------------------+---------------+--------+--------------+
|0.0               |27813          |0.7191  |537.71        |
|1.0               |13639          |0.6977  |860.62        |
|2.0               |4488           |0.7286  |16.06         |
+------------------+---------------+--------+--------------+



In [148]:
fig = px.bar(
    spark_df_para_dicts(acuracia_faixa_altitude_amanha),
    x="faixa_altitude_idx",
    y="accuracy",
    text="accuracy",
    title="Acurácia por faixa de altitude — faixa térmica amanhã",
    labels={
        "faixa_altitude_idx": "Faixa de altitude",
        "accuracy": "Acurácia"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

In [149]:
acuracia_faixa_altitude_semana = spark.sql("""
    SELECT
        faixa_altitude_idx,
        COUNT(*) AS total_registros,
        ROUND(AVG(acertou), 4) AS accuracy,
        ROUND(AVG(altitude), 2) AS altitude_media
    FROM pred_semana_ctx
    WHERE faixa_altitude_idx IS NOT NULL
      AND altitude IS NOT NULL
      AND acertou IS NOT NULL
    GROUP BY faixa_altitude_idx
    ORDER BY faixa_altitude_idx
""")

acuracia_faixa_altitude_semana.createOrReplaceTempView("acuracia_faixa_altitude_semana")

acuracia_faixa_altitude_semana.show(truncate=False)

+------------------+---------------+--------+--------------+
|faixa_altitude_idx|total_registros|accuracy|altitude_media|
+------------------+---------------+--------+--------------+
|0.0               |27813          |0.6106  |537.71        |
|1.0               |13639          |0.6077  |860.62        |
|2.0               |4488           |0.6664  |16.06         |
+------------------+---------------+--------+--------------+



In [150]:
fig = px.bar(
    spark_df_para_dicts(acuracia_faixa_altitude_semana),
    x="faixa_altitude_idx",
    y="accuracy",
    text="accuracy",
    title="Acurácia por faixa de altitude — faixa térmica próximos 7 dias",
    labels={
        "faixa_altitude_idx": "Faixa de altitude",
        "accuracy": "Acurácia"
    }
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

### Insight esperado

Caso os acertos diminuam em altitudes maiores, isso pode indicar que regiões serranas possuem comportamento térmico mais específico.

Mesmo com a variável `altitude` presente no modelo, fatores locais como relevo, cobertura vegetal e massas de ar podem influenciar a temperatura de maneira mais complexa.

## 24. Salvamento das predições, métricas e matrizes de confusão

As saídas do notebook são salvas para comparação posterior com outros modelos.

Como este notebook trata redes neurais como classificação, as métricas salvas são diferentes das métricas de regressão.

In [151]:
predicoes_mlp_amanha_saida = spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        tipo_area_idx,
        macro_regiao_sp_idx,
        faixa_altitude_idx,
        temperatura_amanha,
        classe_temperatura_amanha,
        categoria_real,
        prediction,
        categoria_prevista,
        acertou
    FROM pred_amanha_ctx
""")

predicoes_mlp_semana_saida = spark.sql("""
    SELECT
        station,
        station_code,
        data_formatada,
        tipo_area_idx,
        macro_regiao_sp_idx,
        faixa_altitude_idx,
        temperatura_media_proximos_7_dias,
        classe_temperatura_semana,
        categoria_real,
        prediction,
        categoria_prevista,
        acertou
    FROM pred_semana_ctx
""")

predicoes_mlp_amanha_saida.write.mode("overwrite").parquet(
    f"{resultados_path}/predicoes_mlp_amanha"
)

predicoes_mlp_semana_saida.write.mode("overwrite").parquet(
    f"{resultados_path}/predicoes_mlp_semana"
)

metricas_mlp.write.mode("overwrite").parquet(
    f"{resultados_path}/metricas_redes_neurais_mlp"
)

matriz_confusao_amanha.write.mode("overwrite").parquet(
    f"{resultados_path}/matriz_confusao_mlp_amanha"
)

matriz_confusao_semana.write.mode("overwrite").parquet(
    f"{resultados_path}/matriz_confusao_mlp_semana"
)

print("Predições, métricas e matrizes de confusão salvas com sucesso.")

Predições, métricas e matrizes de confusão salvas com sucesso.


## 25. Salvamento dos modelos treinados

In [152]:
modelo_mlp_amanha.write().overwrite().save(
    f"{modelos_path}/mlp_faixa_termica_amanha"
)

modelo_mlp_semana.write().overwrite().save(
    f"{modelos_path}/mlp_faixa_termica_semana"
)

print("Modelos de redes neurais salvos com sucesso.")

Modelos de redes neurais salvos com sucesso.


## 26. Conclusão

Neste notebook foram treinados dois modelos de **Redes Neurais com PySpark MLlib**:

- um modelo para classificar a faixa térmica de amanhã;
- um modelo para classificar a faixa térmica média dos próximos 7 dias.

Como o MLlib possui rede neural nativa para classificação, o alvo contínuo de temperatura foi convertido em categorias:

- `frio`;
- `ameno`;
- `quente`;
- `muito_quente`.

As bases utilizadas vieram diretamente do pré-processamento, já com:

- split temporal;
- imputação de valores ausentes;
- indexação de variáveis categóricas;
- features temporais de defasagem e janelas móveis.

A avaliação foi feita com:

- Accuracy;
- F1-score;
- Weighted Precision;
- Weighted Recall;
- Matriz de confusão.

Além disso, foram gerados gráficos com Plotly para analisar:

- distribuição das classes;
- comparação geral das métricas;
- matriz de confusão;
- acurácia por tipo de área;
- distribuição das categorias reais por tipo de área;
- acurácia por mês;
- acurácia por macro região;
- relação entre altitude e acerto.

A análise por `tipo_area` permite observar diferenças entre áreas urbanas/metropolitanas, litorâneas, serranas e interiores.

A rede neural não entende sequência temporal automaticamente. Por isso, as features de defasagem e janelas móveis criadas no pré-processamento foram fundamentais para fornecer contexto temporal ao modelo.

Este notebook complementa os modelos de regressão, pois em vez de prever a temperatura exata em °C, classifica a condição térmica futura em faixas interpretáveis.